In [0]:
%pip install google-api-python-client google-auth google-auth-httplib2

In [0]:
%restart_python

In [0]:
%pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib
dbutils.library.restartPython()

In [0]:
# ============================================================================
# RUN CELL — SETUP + BOOTSTRAP (after the one-time setup is complete)
# Loads OAuth credentials from Databricks secrets and creates the destination
# schema and tables if they do not exist yet.
# ============================================================================

# SCOPE_NAME = "gmail-oauth"
SCOPE_NAME = "Unity_Catalog"

try:
    GMAIL_CLIENT_ID = dbutils.secrets.get(catalog="workspace", schema="default", key="client_id")
    GMAIL_CLIENT_SECRET = dbutils.secrets.get(catalog="workspace", schema="default", key="client_secret")
    GMAIL_REFRESH_TOKEN = dbutils.secrets.get(catalog="workspace", schema="default", key="refresh_token")
    print(f"Loaded OAuth credentials from secrets scope '{SCOPE_NAME}'.")
except Exception as e:
    raise RuntimeError(
        f"OAuth credentials not found. Expected secrets scope '{SCOPE_NAME}' with keys "
        "'client_id', 'client_secret' and 'refresh_token'. See the setup markdown cells "
        "above: create the Google OAuth credentials, store them in the scope, then rerun."
    ) from e

# Destination schema (may still be being created elsewhere; IF NOT EXISTS makes this safe)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gmail")

# Messages: one row per Gmail message, keyed on the Gmail message id
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gmail.messages (
  id          STRING NOT NULL COMMENT 'Gmail message id (natural key)',
  thread_id   STRING COMMENT 'Gmail thread id',
  snippet     STRING COMMENT 'Gmail snippet',
  sender      STRING COMMENT 'From header',
  subject     STRING COMMENT 'Subject header',
  body_text   STRING COMMENT 'Plain-text body (base64url-decoded MIME part)',
  received_at TIMESTAMP COMMENT 'Message internalDate in UTC',
  mailbox     STRING COMMENT 'Mailbox address the message was fetched from'
) COMMENT 'Inbox messages polled from the Gmail API (idempotent upsert on id)'
""")

# Classifications: ai_classify result per message id (classified exactly once)
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gmail.message_classifications (
  id             STRING NOT NULL COMMENT 'Gmail message id',
  classification STRING COMMENT 'job_opportunity or other',
  classified_at  TIMESTAMP
) COMMENT 'ai_classify results keyed on message id'
""")

# Alert table: recomputed every run (job_opportunity messages in the last 25 hours)
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gmail.new_job_opportunities (
  id             STRING,
  thread_id      STRING,
  snippet        STRING,
  sender         STRING,
  subject        STRING,
  body_text      STRING,
  received_at    TIMESTAMP,
  mailbox        STRING,
  classification STRING,
  classified_at  TIMESTAMP
) COMMENT 'Job-opportunity messages from the last 25 hours (polled by the SQL alert)'
""")

print("Schema and tables ready:")
for t in ["messages", "message_classifications", "new_job_opportunities"]:
    print(f"  workspace.gmail.{t}")

In [0]:
# ============================================================================
# RUN CELL — WATERMARK
# The last processed received_at (from the messages table itself), or a
# 24-hours-ago default on the very first run.
# ============================================================================

from datetime import datetime, timedelta, timezone

row = spark.sql("SELECT MAX(received_at) AS watermark FROM workspace.gmail.messages").collect()[0]
if row["watermark"] is not None:
    watermark = row["watermark"].astimezone(timezone.utc)
    source = "messages table"
else:
    watermark = datetime.now(timezone.utc) - timedelta(days=1)
    source = "default (first run: last 24 hours)"

print(f"Watermark: {watermark.isoformat()} (source: {source})")

In [0]:
# ============================================================================
# RUN CELL — FETCH NEW MESSAGES FROM GMAIL
# Lists inbox messages received after the watermark, then fetches metadata +
# plain-text body for each. Gmail search only supports day granularity, so we
# over-fetch with `after:` and filter precisely on internalDate — re-running is
# safe and no message is skipped or duplicated.
# ============================================================================

import base64
from datetime import datetime, timezone

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

creds = Credentials(
    token=None,
    refresh_token=GMAIL_REFRESH_TOKEN,
    client_id=GMAIL_CLIENT_ID,
    client_secret=GMAIL_CLIENT_SECRET,
    token_uri="https://oauth2.googleapis.com/token",
    scopes=["https://www.googleapis.com/auth/gmail.readonly"],
)
creds.refresh(Request())
gmail = build("gmail", "v1", credentials=creds)

MAILBOX = gmail.users().getProfile(userId="me").execute()["emailAddress"]
print(f"Mailbox: {MAILBOX}")

after_date = watermark.strftime("%Y/%m/%d")
q = f"in:inbox after:{after_date}"

candidate_ids = []
page_token = None
while True:
    resp = gmail.users().messages().list(
        userId="me", q=q, maxResults=500, pageToken=page_token
    ).execute()
    candidate_ids += [m["id"] for m in resp.get("messages", [])]
    page_token = resp.get("nextPageToken")
    if not page_token:
        break
print(f"{len(candidate_ids)} candidate message(s) matching {q!r}; filtering after the watermark...")

def header(msg, name):
    for h in msg["payload"].get("headers", []):
        if h["name"].lower() == name.lower():
            return h["value"]
    return None

def decode_plain_text(payload):
    """Return the first non-empty text/plain part, walking nested MIME parts."""
    if payload.get("mimeType") == "text/plain":
        data = payload.get("body", {}).get("data")
        if data:
            return base64.urlsafe_b64decode(data).decode("utf-8", "replace")
    for part in payload.get("parts") or []:
        text = decode_plain_text(part)
        if text:
            return text
    return ""

new_messages = []
for mid in candidate_ids:
    msg = gmail.users().messages().get(userId="me", id=mid, format="full").execute()
    internal_ms = msg.get("internalDate")
    received_at = (
        datetime.fromtimestamp(int(internal_ms) / 1000, tz=timezone.utc)
        if internal_ms else None
    )
    if received_at is None or received_at <= watermark:
        continue
    new_messages.append({
        "id": msg["id"],
        "thread_id": msg["threadId"],
        "snippet": msg.get("snippet"),
        "sender": header(msg, "From"),
        "subject": header(msg, "Subject"),
        "body_text": decode_plain_text(msg["payload"]) or msg.get("snippet"),
        "received_at": received_at,
        "mailbox": MAILBOX,
    })

print(f"{len(new_messages)} new message(s) after the watermark.")

In [0]:
# ============================================================================
# RUN CELL — UPSERT INTO DELTA (idempotent: keyed on the Gmail message id)
# ============================================================================

if new_messages:
    df = spark.createDataFrame(new_messages)  # dict keys match the table columns by name
    df.createOrReplaceTempView("gmail_new_messages")
    spark.sql("""
      MERGE INTO workspace.gmail.messages AS t
      USING gmail_new_messages AS s
      ON t.id = s.id
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Upserted {len(new_messages)} message(s) into workspace.gmail.messages.")
else:
    print("No new messages — nothing to upsert.")

In [0]:
# ============================================================================
# RUN CELL — CLASSIFY NEW MESSAGES WITH ai_classify
# Classifies only messages that have not been classified yet; results are
# stored once per message id so each message is classified exactly once.
# ============================================================================

spark.sql("""
  MERGE INTO workspace.gmail.message_classifications AS c
  USING (
    SELECT
      m.id,
      ai_classify(
        concat_ws(' | ', coalesce(m.subject, ''), coalesce(m.body_text, coalesce(m.snippet, ''))),
        array('job_opportunity', 'other')
      ) AS classification,
      current_timestamp() AS classified_at
    FROM workspace.gmail.messages m
    LEFT ANTI JOIN workspace.gmail.message_classifications x ON x.id = m.id
    WHERE m.received_at > current_timestamp() - INTERVAL 25 HOURS
  ) AS s
  ON c.id = s.id
  WHEN NOT MATCHED THEN INSERT *
""")
print("Classification updated.")

In [0]:
# ============================================================================
# RUN CELL — REFRESH THE ALERT TABLE
# Recomputes workspace.gmail.new_job_opportunities each run: every message
# classified 'job_opportunity' received in the last 25 hours. The SQL alert
# polls this table on each of its evaluations.
# ============================================================================

spark.sql("""
  CREATE OR REPLACE TABLE workspace.gmail.new_job_opportunities AS
  SELECT
    m.id, m.thread_id, m.snippet, m.sender, m.subject, m.body_text,
    m.received_at, m.mailbox, c.classification, c.classified_at
  FROM workspace.gmail.messages m
  JOIN workspace.gmail.message_classifications c ON c.id = m.id
  WHERE c.classification = 'job_opportunity'
    AND m.received_at > current_timestamp() - INTERVAL 25 HOURS
""")

n = spark.sql(
    "SELECT COUNT(*) AS n FROM workspace.gmail.new_job_opportunities"
).collect()[0]["n"]
print(f"workspace.gmail.new_job_opportunities refreshed: {n} job-opportunity message(s) in the last 25 hours.")